# Global Signal Limits from Baryon Power Spectrum

Compute dark photon constraints using the global (sky-averaged) 21cm signal. Uses CLASS power spectra to predict conversion probabilities and derive limits on the dark photon-photon coupling ε.

## Theory

The conversion probability depends on the baryon density field through $m_\gamma^2 = \kappa \rho_b$. By modeling density fluctuations with lognormal statistics and known $P_{bb}(k,z)$, we can predict:

- Total conversion probability vs dark photon mass
- Constraints from requiring global signal deviations < observational limits

This is heavily based on code from https://github.com/smsharma/dark-photons-perturbations, and the user needs to install this code first and specify its path using `grf_path`. 

## Setup

In [ ]:
import sys
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec, ticker
from scipy.optimize import fsolve
import seaborn as sns
import numpy as np
from scipy.integrate import cumulative_trapezoid, quad
from tqdm import *
from scipy.interpolate import interp1d
import scipy.special as sp

# Dark photon perturbation theory modules
grf_path = "/home/bakerem/dark-photons-perturbations"
sys.path.append(grf_path)

from grf.grf import PerturbedProbability, FIRAS
from grf.pk_interp import PowerSpectrumGridInterpolator
from grf.units import *

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Custom plotting settings
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Power Spectrum Setup and Mass Grid

In [ ]:
# Initialize with linear baryon power spectrum
pspec_lin_baryon = PowerSpectrumGridInterpolator("lin_baryon")
prob = FIRAS(pspec_lin_baryon)

# Density fluctuation cutoff for lognormal model validity
one_plus_delta_bound = 1e2  # δ < 100 (conservative bound)

# Dark photon mass grids for different regimes
low_mA_list = np.geomspace(1.5e-14, 1e-13, 15)   # Low mass (recombination era)
halo_mA_list = np.geomspace(1e-13, 1e-11, 25)    # Halo mass range
mA_list = np.concatenate([halo_mA_list, low_mA_list]) * eV

# Extended mass range for comprehensive survey
mA_list = np.geomspace(1e-16, 1e-9, 300) * eV

## Conversion Probability Calculation

In [ ]:
# Initialize arrays for total conversion probabilities
P_tot_an_ary = np.zeros_like(mA_list)

# Redshift array (exclude EoR to avoid 21cmFAST complications)
rs_array = np.geomspace(0.001, 1000, 10000)
rs_array = rs_array[(rs_array < 5) | (rs_array > 35)]  # Skip 5 < z < 35

# Conversion probabilities with density fluctuation bound
P_tot_bounded_ary = np.zeros_like(mA_list)

print("Computing conversion probabilities for lognormal density field...")

for i_m_Ap, m_Ap in enumerate(tqdm(mA_list, desc="Dark photon masses")):
    # Physical parameters at z=17 (post-recombination reference)
    T0 = 2.73 * 8.62e-5  # CMB temperature in eV
    xobs = 0.0251/(1+17)  # Free electron fraction at z=17
    omega_ary = xobs * T0 * eV  # Plasma frequency
    
    # Compute dP/dz using lognormal PDF with density cutoff
    dPdrs_array = prob._dP_dz(
        z_ary=rs_array, 
        m_Ap=m_Ap, 
        k_min=1e-3, k_max=1e3,  # Integration bounds
        omega=omega_ary, 
        pdf='lognormal', 
        one_plus_delta_bound=one_plus_delta_bound
    )[0][0]
    
    # Integrate over redshift to get total probability
    P_tot_bounded_ary[i_m_Ap] = np.trapz(dPdrs_array, rs_array)

print(f"Conversion probabilities computed for {len(mA_list)} masses")

## Coupling Limits from Global Signal

In [ ]:
def find_nearest(array, value):
    """Find nearest array element to target value."""
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

# Observational constraint: global signal deviation limit
bt_limit = -100  # mK (conservative bound)
z = 17  # Reference redshift (post-recombination)
Tgamma0_mK = 2.73 * 1000  # CMB monopole in mK

# Find reference redshift in array
z_index = np.where(rs_array == find_nearest(rs_array, z))[0]
bt_at_z = 0  # Fiducial ΛCDM brightness temperature
xobs = 0.0251/(1+z)  # Free electron fraction
xe = prob.x_e(rs_array)  # Ionization history

print("Computing coupling limits using lognormal conversion probabilities...")

# Solve for coupling limit at each mass
lognorm_lims = []
for i_m_Ap, m_Ap in enumerate(mA_list):
    def find_lognorm_eps(eps):
        """
        Find coupling that produces bt_limit deviation in global signal.
        
        Global signal change: ΔT = -T_γ₀ * P_total * ε²
        """
        P_lognorm = P_tot_bounded_ary[i_m_Ap] * eps**2 
        return bt_limit - (bt_at_z - Tgamma0_mK * P_lognorm)
    
    # Solve for critical coupling
    lognorm_sol = fsolve(find_lognorm_eps, 1e-6, maxfev=10000, xtol=1e-10, full_output=True)
    
    # Check if solution converged
    if lognorm_sol[2] != 1 or lognorm_sol[0] > 1:
        lognorm_eps = 100  # No meaningful constraint
    else:
        lognorm_eps = lognorm_sol[0][0]
    
    lognorm_lims.append(np.array([m_Ap/eV, lognorm_eps]))

lognorm_lims = np.array(lognorm_lims)
print(f"Coupling limits computed for {len(mA_list)} masses")

## Homogeneous Universe Comparison

In [ ]:
# Compute limits assuming homogeneous (smooth) universe for comparison
print("Computing homogeneous universe limits...")

homo_lims = []
for i_m_Ap, m_Ap in enumerate(mA_list):    
    mA = m_Ap / eV
    
    # Plasma mass evolution in homogeneous universe
    mgamma2 = prob.m_A_sq(rs_array, omega_ary) / eV**2
    
    # Physical constants
    T0 = 2.73 * 8.62e-5  # CMB temperature in eV
    H0 = 67.66 * 6.57895e-16 * 3.241e-20  # Hubble constant conversion
    
    # Find redshift where m_γ = m_A (conversion occurs)
    mgamma2_interp = interp1d(rs_array, mgamma2, bounds_error=False, fill_value="extrapolate")
    
    # Initial guess for solver
    if mA < 1.5e-13 and mA > np.sqrt(mgamma2[0]):
        x0 = 1
    else:
        x0 = 100
    
    # Solve for conversion redshift
    output = fsolve(lambda z: np.sqrt(mgamma2_interp(z)) - mA, x0, maxfev=10000, full_output=True)
    z_star = output[0]
    
    # Check if solution is valid
    if output[2] != 1 or z_star > rs_array.max():
        Ptot_prefac = 1e-100  # No conversion possible
    else:
        # Homogeneous conversion probability formula
        Ptot_prefac = (np.pi * mA**2 / 
                      (3 * xobs * T0 * (1+z_star[0]) * 
                       prob.cosmo.H(z_star).value[0] * Kmps / Mpc / eV))   
     
    # Solve for coupling limit
    homo_eps = np.sqrt(-(bt_limit - bt_at_z) / (Tgamma0_mK * Ptot_prefac))
    homo_lims.append([mA, homo_eps])

homo_lims = np.array(homo_lims)
print("Homogeneous limits computed")


## Final Limits Plot

In [ ]:
# Load FIRAS comparison data
m_Ap_DP, lim_DP = np.transpose(np.loadtxt("/home/bakerem/dark_photon_21cm_constraints/halo_data/data_from_papers/fiducial_DP_FIRAS_one_plus_delta_1e2.csv", skiprows=2, delimiter=','))

# Transition between lognormal and homogeneous regimes
# Switch to homogeneous at high masses where fluctuations become irrelevant
mA_index = np.where(homo_lims[:,0] == find_nearest(homo_lims[:,0], np.sqrt(prob.m_A_sq(370, 1)/eV**2)))[0][0]

plot_lognorm_lims = lognorm_lims.copy()
plot_lognorm_lims[mA_index:, 1] = homo_lims[mA_index:, 1]  # Use homogeneous at high masses
np.save("../halo_data/limits/21cm_global_lognorm_lims.npy", plot_lognorm_lims)
# Create exclusion plot
plt.plot(plot_lognorm_lims[:,0], plot_lognorm_lims[:,1], label="Lognormal", linewidth=2)

# Show existing FIRAS constraint region
plt.fill_between(m_Ap_DP, lim_DP, 1e-2, alpha=0.2, color=cols_default[3])

plt.xscale("log")
plt.yscale("log")
plt.ylim(1e-8, 1e-2)
plt.xlim(1e-15, 5e-10)

plt.xlabel(r"$m_{A'}$ [eV]")
plt.ylabel(r"$\varepsilon$")
plt.legend(loc="lower left")

# Note: FIRAS region shown for comparison